In [1]:
#   libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import r2_score, accuracy_score, confusion_matrix, roc_curve

from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder

## 1. Data Loading and Initial Exploration

This section loads the dataset and provides a first look at the data structure and content. The dataset contains various factors influencing student performance, with 'Exam_Score' as the primary target variable.

In [2]:
# Load Dataset

df = pd.read_csv("StudentPerformanceFactors.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'StudentPerformanceFactors.csv'

### 2.1 Simple Linear Regression: Predicting Exam Score using Hours Studied

We first build a simple linear regression model to predict `Exam_Score` based solely on `Hours_Studied`. This helps us understand the basic correlation between study time and exam performance.

## 2. Linear Regression Models

This section explores the relationship between various factors and student `Exam_Score` using linear regression. We will start with a simple linear regression and then move to a multiple linear regression model.

The `df.head()` output above shows the first few rows of the dataset, giving an overview of the features available, such as `Hours_Studied`, `Attendance`, `Parental_Involvement`, `Exam_Score`, etc.

In [ ]:
# Using Hours_Studied to predict Exam_Score

X = df[['Hours_Studied']]
y = df['Exam_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

model = LinearRegression()
model.fit(X_train, y_train)

# Results

y_pred = model.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred))

### 2.2 Multiple Linear Regression: Predicting Exam Score with Multiple Factors

Next, we extend the model to include more predictive features, such as `Hours_Studied`, `Attendance`, `Sleep_Hours`, `Previous_Scores`, and `Motivation_Level` (one-hot encoded). This allows for a more comprehensive prediction of `Exam_Score`.

The R2 Score of approximately 0.20 suggests that `Hours_Studied` alone explains about 20% of the variance in `Exam_Score`. This indicates a positive but not very strong linear relationship.

In [ ]:
X = df[['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores']]
X = pd.concat([X, pd.get_dummies(df['Motivation_Level'], prefix='Motivation_Level')], axis=1)
y = df['Exam_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluate
print("R2 Score (Multiple):", r2_score(y_test, y_pred))

### 3.1 Categorizing Exam Scores

We convert the numerical `Exam_Score` into three performance categories: 'Low' (0-50), 'Medium' (51-75), and 'High' (76-100). This transformation allows us to use classification models.

## 3. Classification Models

To better understand student performance in categories, we will convert the continuous `Exam_Score` into categorical performance levels. We will then apply various classification algorithms.

With multiple features, the R2 Score significantly increases to approximately 0.59. This indicates that these additional factors collectively explain about 59% of the variance in `Exam_Score`, making it a much better predictive model than `Hours_Studied` alone.

In [ ]:
# Convert Exam Score into categories (Low, Medium, High)

df['Score_Category'] = pd.cut(df['Exam_Score'],
                             bins=[0, 50, 75, 100],
                             labels=['Low', 'Medium', 'High'])

### 3.2 Logistic Regression for Performance Level Prediction

This section uses Logistic Regression to classify student performance into 'Low', 'Medium', or 'High' based on selected features. We prepare the data by selecting relevant features and creating dummy variables for categorical features.

In [ ]:
# Select data

X_features = df[['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores']]
X_motivation = pd.get_dummies(df['Motivation_Level'], prefix='Motivation_Level')

X_combined = pd.concat([X_features, X_motivation], axis=1)
y_logreg = df['Score_Category'] # Rename y to y_logreg to avoid conflict

# Combine X and y, then drop rows with any NaN values
combined_df = pd.concat([X_combined, y_logreg], axis=1)
combined_df.dropna(inplace=True) # Drop rows if any column has NaN

X_logreg = combined_df.drop('Score_Category', axis=1) # Rename X to X_logreg
y_logreg = combined_df['Score_Category']

# split data
X_train_logreg, X_test_logreg, y_train_logreg, y_test_logreg = train_test_split(X_logreg, y_logreg, test_size=0.2, random_state=1)

In [ ]:
# Train Logistic Regression model
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_logreg, y_train_logreg)

# Predict and evaluate
y_pred_log = log_model.predict(X_test_logreg)
print("Logistic Accuracy:", accuracy_score(y_test_logreg, y_pred_log))

The `Score_Category` column has been created and is now available in the DataFrame. We use this for classification tasks.

The Logistic Regression model achieved an accuracy of approximately 99%, indicating excellent performance in classifying student performance levels.

In [ ]:
# add small noise so points spread
y_test_jitter = y_test + np.random.uniform(-0.1, 0.1, size=len(y_test))
y_pred_jitter = y_pred_log + np.random.uniform(-0.1, 0.1, size=len(y_pred_log))

plt.figure(figsize=(10,5))

plt.scatter(range(len(y_test)), y_test_jitter, label="Actual", alpha=0.6)
plt.scatter(range(len(y_pred_log)), y_pred_jitter, label="Predicted", alpha=0.6)

plt.yticks([0,1,2], ['Low','Medium','High'])
plt.legend()
plt.title("Comparison of Actual and Predicted Student Performance Levels using Logistic Regression")
plt.xlabel("Student Index")
plt.ylabel("Performance Level (Low, Medium, High)")

plt.show()

This scatter plot visually compares the actual and predicted student performance levels. The jitter effect is applied to better visualize overlapping points. The strong alignment of predicted points with actual points further confirms the high accuracy of the logistic regression model.

### 3.3 K-Means Clustering: Student Performance Groups

K-Means clustering is used to identify natural groupings of students based on their `Hours_Studied` and `Exam_Score`. This helps in understanding different student profiles.

The confusion matrix provides a detailed breakdown of the model's predictions. The high values along the diagonal (correct classifications) and very low values off-diagonal (misclassifications) demonstrate the model's effectiveness in distinguishing between 'Low', 'Medium', and 'High' performance categories.

In [ ]:
# Cluster Graph using K-Means

import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Select two variables for easy visual explanation
cluster_data = df[['Hours_Studied', 'Exam_Score']]

# Scale the data
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_data)

# Create 3 clusters
kmeans = KMeans(n_clusters=3, random_state=1, n_init=10)
df['Cluster'] = kmeans.fit_predict(cluster_scaled)

# Draw cluster graph
plt.figure(figsize=(8, 6))
plt.scatter(
    df['Hours_Studied'],
    df['Exam_Score'],
    c=df['Cluster']
)
plt.xlabel("Hours Studied")
plt.ylabel("Exam Score")
plt.title("K-Means Cluster Graph: Hours Studied vs Exam Score")
plt.show()

# Show average values for each cluster
cluster_summary = df.groupby('Cluster')[['Hours_Studied', 'Exam_Score']].mean()
print(cluster_summary)

### 3.4 K-Nearest Neighbors (KNN) Classification

KNN is a non-parametric, instance-based learning algorithm. This section trains a KNN model to classify student performance and evaluates its accuracy across different `K` values to find the optimal number of neighbors.

The cluster graph shows three distinct groups of students based on their hours studied and exam scores. The summary table below highlights the average `Hours_Studied` and `Exam_Score` for each cluster, revealing different performance patterns.

*   **Cluster 0:** Students with lower hours studied and lower exam scores.
*   **Cluster 1:** Students with higher hours studied and higher exam scores.
*   **Cluster 2:** Students with moderate hours studied and moderate exam scores.

In [ ]:
k_values = range(1, 15)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = knn.score(X_test, y_test)
    accuracies.append(acc)

plt.plot(k_values, accuracies, marker='o')
plt.title("KNN Accuracy vs K Value")
plt.xlabel("K")
plt.ylabel("Accuracy")
plt.show()

### 3.5 Naive Bayes Classification

Naive Bayes is a probabilistic classifier based on Bayes' theorem, assuming independence between features. This section applies a Gaussian Naive Bayes model to predict student performance levels.

The plot shows how the KNN model's accuracy changes with different values of `K`. We can observe that the accuracy remains consistently high across a range of `K` values, suggesting the model is robust.

In [ ]:
# Train Naive Bayes model
nb = GaussianNB()
nb.fit(X_train, y_train)

# Predict and evaluate
y_pred = nb.predict(X_test)
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred))

The Naive Bayes model also shows very high accuracy, close to 99.8%, indicating its strong predictive capability for this dataset.

In [ ]:
nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

cm = confusion_matrix(y_test, y_pred_nb)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title("Naive Bayes Confusion Matrix")
plt.show()

### 3.6 Decision Tree Classification

Decision Trees are tree-like models where each internal node represents a test on a feature, each branch represents the outcome of the test, and each leaf node represents a class label. This section builds and visualizes a Decision Tree to predict student performance levels.

The confusion matrix for the Naive Bayes model confirms its high accuracy, with most predictions falling on the main diagonal. This indicates that the model is performing very well in classifying the performance levels.

In [ ]:
# Decision Tree Graph

df = df.dropna()

# Create performance categories from Exam_Score
df['Performance_Level'] = pd.cut(
    df['Exam_Score'],
    bins=[0, 65, 75, 100],
    labels=['Low', 'Medium', 'High']
)

# Select simple features that are easy to explain
X = df[['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores']]
y = df['Performance_Level']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# Train decision tree
tree_model = DecisionTreeClassifier(max_depth=3, random_state=1)
tree_model.fit(X_train, y_train)

# Predict and check accuracy
y_pred = tree_model.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))

# Draw tree graph
plt.figure(figsize=(18, 8))
plot_tree(
    tree_model,
    feature_names=X.columns,
    class_names=['Low', 'Medium', 'High'],
    filled=True,
    rounded=True
)
plt.title("Decision Tree for Student Performance Level")
plt.show()

## 4. Additional Data Visualizations

This section includes additional visualizations to understand the dataset better, including scatter plots, histograms, and class distributions.

The Decision Tree achieved an accuracy of approximately 80.5%. The visualization of the tree above provides insights into the decision rules learned by the model, showing how different feature values lead to predictions of 'Low', 'Medium', or 'High' performance levels.

In [ ]:
# Select features for clustering
X = df[['Hours_Studied', 'Attendance', 'Exam_Score']]

# Apply K-Means with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=1)

# Assign cluster labels
df['Cluster'] = kmeans.fit_predict(X)

# Show results
df[['Hours_Studied', 'Attendance', 'Exam_Score', 'Cluster']].head()

### 4.2 Scatter Plot: Hours Studied vs. Exam Score

This plot visualizes the direct relationship between `Hours_Studied` and `Exam_Score`, helping to quickly identify any visible correlations or patterns.

The head of the DataFrame with the new 'Cluster' column shows how students are grouped based on `Hours_Studied`, `Attendance`, and `Exam_Score`. This can reveal different behavioral segments among students.

In [ ]:
plt.scatter(df['Hours_Studied'], df['Exam_Score'])
plt.title("Hours Studied vs Exam Score")
plt.xlabel("Hours Studied")
plt.ylabel("Exam Score")
plt.show()

### 4.3 Histogram: Distribution of Exam Scores

This histogram illustrates the frequency distribution of `Exam_Score`, showing how scores are spread across different ranges.

This scatter plot shows a general trend where students who study more hours tend to achieve higher exam scores, although there is considerable variability.

The histogram indicates that most students score in the medium to high range, with fewer students achieving very low or very high scores, suggesting a somewhat normal distribution around the average performance.

In [ ]:
df['Performance_Level'].value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.show()

### 4.5 Logistic Regression Feature Importance

This plot visualizes the coefficients of the logistic regression model, indicating the importance and direction of influence of each feature on predicting student performance levels.

This bar plot clearly shows the distribution of students across the three performance levels. It appears there is an imbalance, with a larger number of students in the 'Medium' category compared to 'Low' and 'High'.

In [ ]:
importance = log_model.coef_[0]

plt.bar(X_logreg.columns, importance)
plt.title("Logistic Regression Feature Importance")
plt.xticks(rotation=45)
plt.show()